In [ ]:
import pandas as pd

# ========= 1) Excel 读列 =========
file_path = r"your_data.xlsx"
sheet_name = "Outlier"
series_col = "value"
df = pd.read_excel(file_path, sheet_name=sheet_name)
s = df[series_col]

# ========= 2) 参数模板 =========
params = {
    "sigma_k": 3.0  # float: 3σ中的倍数，默认3
}

mu, sigma = s.mean(), s.std()
outlier_3sigma = (s < mu - params["sigma_k"]*sigma) | (s > mu + params["sigma_k"]*sigma)

q1, q3 = s.quantile(0.25), s.quantile(0.75)
iqr = q3 - q1
outlier_box = (s < q1 - 1.5*iqr) | (s > q3 + 1.5*iqr)

print("3σ异常数量:", outlier_3sigma.sum())
print("箱线图异常数量:", outlier_box.sum())


In [ ]:
"""
3σ 原则、箱型图检测异常值

使用方法：
1. 按照下方 TODO 修改 DATA_FILE、列名、参数和输出文件名。
2. 将数据文件放在本脚本同目录，或把 DATA_FILE 改成绝对路径。
3. 运行：python "3σ 原则、箱型图检测异常值.py"
"""

from pathlib import Path
import numpy as np
import pandas as pd



DATA_FILE = "data.csv"  # TODO: 请填写[数据文件路径]，说明：CSV/Excel 均可；若使用 Excel，请在 load_data 中改为 read_excel。
OUTPUT_FILE = "model_output.csv"  # TODO: 请填写[输出文件名]，说明：保存模型结果，建议保留 .csv 或 .xlsx 后缀。
RANDOM_STATE = 42  # TODO: 请填写[随机种子]，说明：用于复现实验；整数即可。
SIGMA_K = 3  # TODO: 请填写[标准差倍数]，说明：正态近似下常用 3。
IQR_K = 1.5  # TODO: 请填写[IQR 倍数]，说明：箱型图常用 1.5，严格检测可取 1.0。



REQUIRES_DATA = True  # 参数型模型可不提供数据文件；表格型模型必须提供数据。


def load_data() -> pd.DataFrame:
    """读取用户数据；竞赛时通常把 Excel/CSV 表格整理成一行一个样本。"""
    path = Path(DATA_FILE)
    if not path.exists():
        if not REQUIRES_DATA:
            return pd.DataFrame()
        raise FileNotFoundError(
            f"未找到数据文件 {DATA_FILE}。请先修改 DATA_FILE，或将数据放到脚本同目录。"
        )
    if path.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(path)
    return pd.read_csv(path)


def run_model(data: pd.DataFrame) -> None:
    numeric = data.select_dtypes(include=[np.number])
    flags = pd.DataFrame(index=data.index)
    for col in numeric.columns:
        mean, std = numeric[col].mean(), numeric[col].std(ddof=1)
        q1, q3 = numeric[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        flags[f"{col}_3sigma异常"] = (numeric[col] < mean - SIGMA_K * std) | (numeric[col] > mean + SIGMA_K * std)
        flags[f"{col}_箱型图异常"] = (numeric[col] < q1 - IQR_K * iqr) | (numeric[col] > q3 + IQR_K * iqr)
    pd.concat([data, flags], axis=1).to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
    print(flags.sum())


if __name__ == "__main__":
    df = load_data()
    run_model(df)